# 05 Model Training & Multi-Failure Mode Evaluation

This notebook trains and evaluates models for predicting both overall machine failure (`machine_failure`) and specific failure modes (`twf`, `hdf`, `pwf`, `osf`, `rnf`).

## 1. Import Libraries & Setup

In [2]:
import os
import sys
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, f1_score, recall_score, precision_score, roc_auc_score

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Define model export directory
model_dir = '../models'
if not os.path.exists(model_dir):
    model_dir = 'models'
os.makedirs(model_dir, exist_ok=True)

## 2. Load Processed Datasets & Target Setup

In [3]:
data_dir = '../data/processed'
if not os.path.exists(data_dir):
    data_dir = 'data/processed'

X_train = pd.read_csv(os.path.join(data_dir, 'X_train.csv'))
X_test  = pd.read_csv(os.path.join(data_dir, 'X_test.csv'))
y_train_full = pd.read_csv(os.path.join(data_dir, 'y_train.csv'))
y_test_full  = pd.read_csv(os.path.join(data_dir, 'y_test.csv'))

target_cols = ['machine_failure', 'twf', 'hdf', 'pwf', 'osf', 'rnf']
y_train = y_train_full[target_cols]
y_test  = y_test_full[target_cols]

print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")
print("Target value counts in Training Set:")
print(y_train.sum())
print("\nTarget value counts in Testing Set:")
print(y_test.sum())

X_train shape: (8000, 9) | X_test shape: (2000, 9)
Target value counts in Training Set:
machine_failure    271
twf                 36
hdf                 86
pwf                 82
osf                 82
rnf                 15
dtype: int64

Target value counts in Testing Set:
machine_failure    68
twf                10
hdf                29
pwf                13
osf                16
rnf                 4
dtype: int64


## 3. Per-Target Scale-Weighted Model Training & Evaluation

In [4]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),       columns=X_test.columns)

models_dict = {}
eval_results = []

for col in target_cols:
    pos_cnt = (y_train[col] == 1).sum()
    neg_cnt = (y_train[col] == 0).sum()
    w = neg_cnt / max(1, pos_cnt)
    
    if col == 'machine_failure':
        clf = LGBMClassifier(n_estimators=150, learning_rate=0.05, max_depth=6, scale_pos_weight=w, random_state=RANDOM_STATE, verbose=-1)
        thresh = 0.5
    elif col in ['hdf', 'pwf', 'osf']:
        clf = LGBMClassifier(n_estimators=150, learning_rate=0.05, max_depth=6, scale_pos_weight=w, random_state=RANDOM_STATE, verbose=-1)
        thresh = 0.5
    elif col == 'twf':
        clf = LGBMClassifier(n_estimators=200, learning_rate=0.03, max_depth=6, scale_pos_weight=w*0.8, random_state=RANDOM_STATE, verbose=-1)
        thresh = 0.3
    else: # rnf
        clf = LGBMClassifier(n_estimators=100, learning_rate=0.05, max_depth=4, scale_pos_weight=w*0.5, random_state=RANDOM_STATE, verbose=-1)
        thresh = 0.2
        
    clf.fit(X_train_scaled, y_train[col])
    models_dict[col] = clf
    
    probas = clf.predict_proba(X_test_scaled)[:, 1]
    preds = (probas >= thresh).astype(int)
    
    eval_results.append({
        'Target': col.upper(),
        'Recall': round(recall_score(y_test[col], preds, zero_division=0), 4),
        'Precision': round(precision_score(y_test[col], preds, zero_division=0), 4),
        'F1-Score': round(f1_score(y_test[col], preds, zero_division=0), 4),
        'ROC-AUC': round(roc_auc_score(y_test[col], probas), 4)
    })

eval_df = pd.DataFrame(eval_results).set_index('Target')
display(eval_df)

,Recall,Precision,F1-Score,ROC-AUC
Target,,,,
MACHINE_FAILURE,0.8676,0.6629,0.7516,0.9850
TWF,0.3000,0.0652,0.1071,0.8609
HDF,1.0000,1.0000,1.0000,1.0000
PWF,1.0000,0.9286,0.9630,0.9999
OSF,1.0000,0.8889,0.9412,1.0000
RNF,0.2500,0.0046,0.0091,0.5358


## 4. Multi-Output Pipeline Model Training & Export

In [5]:
failure_cols = ['twf', 'hdf', 'pwf', 'osf', 'rnf']
lgbm_base = LGBMClassifier(n_estimators=100, learning_rate=0.05, max_depth=6, random_state=RANDOM_STATE, verbose=-1)
model_multi = MultiOutputClassifier(lgbm_base)
model_multi.fit(X_train_scaled, y_train[failure_cols])

joblib.dump(model_multi, os.path.join(model_dir, 'multi_label_model.pkl'))
joblib.dump(scaler, os.path.join(model_dir, 'scaler.pkl'))
print(f"Models and scaler saved successfully to {model_dir}/")

Models and scaler saved successfully to ../models/
